In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from src.utils.db import get_engine

engine = get_engine()

# hackle_events 테이블 전처리

In [6]:
event_df = pd.read_sql(
    "SELECT * FROM hackle_events",
    engine
)

In [9]:
event_df.head()

,event_id,event_datetime,event_key,session_id,id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,2023-07-18 19:40:17,$session_start,4OzYh3seq3VKytpSn5pvQkZNQii1,00000533-3f1c-4b3b-81f1-0c8f35754b4e,,,NaN,NaN,NaN,NaN
1,00000716-27e9-4e72-a602-d0ce61784b06,2023-07-18 21:07:24,click_question_open,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,00000716-27e9-4e72-a602-d0ce61784b06,,,64.0,436.0,4830.0,NaN
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,2023-08-06 20:18:03,click_bottom_navigation_profile,6bcea65d-9f40-46fc-888c-700fe707483f,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,,,26.0,174.0,4729.0,NaN
3,00000981-5e2a-4111-993e-4f1891ad9a53,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,00000981-5e2a-4111-993e-4f1891ad9a53,,,61.0,44.0,142.0,NaN
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,2023-07-24 15:03:37,click_bottom_navigation_lab,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,00000a7a-ba72-4332-b4a9-7910670aaeb2,,,119.0,545.0,3287.0,NaN


In [7]:
event_df.shape

(11441319, 11)

In [8]:
event_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11441319 entries, 0 to 11441318
Data columns (total 11 columns):
 #   Column          Dtype         
---  ------          -----         
 0   event_id        str           
 1   event_datetime  datetime64[us]
 2   event_key       str           
 3   session_id      str           
 4   id              str           
 5   item_name       str           
 6   page_name       str           
 7   friend_count    float64       
 8   votes_count     float64       
 9   heart_balance   float64       
 10  question_id     float64       
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 960.2 MB


## 결측값 처리

In [10]:
event_df.isna().sum()

event_id                 0
event_datetime           0
event_key                0
session_id               0
id                       0
item_name                0
page_name                0
friend_count        752556
votes_count         754554
heart_balance       728643
question_id       10991835
dtype: int64

In [12]:
for col in ["item_name", "page_name"]:
    event_df[col] = event_df[col].replace(r"^\s*$", pd.NA, regex=True)

In [13]:
event_df.isna().sum()

event_id                 0
event_datetime           0
event_key                0
session_id               0
id                       0
item_name         11428280
page_name         10652540
friend_count        752556
votes_count         754554
heart_balance       728643
question_id       10991835
dtype: int64

In [21]:
null_summary = pd.DataFrame({
    "null_count": event_df.isna().sum(),
    "null_ratio": (event_df.isna().mean() * 100).round(2)
})

null_summary

,null_count,null_ratio
event_id,0,0.00
event_datetime,0,0.00
event_key,0,0.00
session_id,0,0.00
id,0,0.00
item_name,11428280,99.89
page_name,10652540,93.11
friend_count,752556,6.58
votes_count,754554,6.59
heart_balance,728643,6.37


---

In [23]:
event_df.loc[
    event_df["item_name"].notna(),
    "event_key"
].value_counts()

event_key
click_purchase    13039
Name: count, dtype: int64

In [14]:
event_df[~event_df["item_name"].isna()].head()

,event_id,event_datetime,event_key,session_id,id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
1049,00063238-57cd-4cb5-901e-73fbf0620307,2023-08-07 13:46:19,click_purchase,0f764eda-71d7-416d-ae88-546fdf84caa9,00063238-57cd-4cb5-901e-73fbf0620307,777 하트,NaN,24.0,29.0,508.0,NaN
1380,00082bfc-acec-43df-bd3c-81771e2f8813,2023-07-28 18:43:21,click_purchase,16504bf1-6ef0-4bb2-bd6a-5bc8d3f3072e,00082bfc-acec-43df-bd3c-81771e2f8813,무료충전소,NaN,0.0,0.0,73.0,NaN
2350,000e1ff0-fcd7-4cee-b107-29f681f42c59,2023-08-06 21:10:30,click_purchase,F5CF51C8-6251-4C34-9357-94B5E850E920,000e1ff0-fcd7-4cee-b107-29f681f42c59,777 하트,NaN,38.0,18.0,383.0,NaN
2566,000f4c25-4e91-40d4-aaa1-b885692a0556,2023-07-21 00:56:25,click_purchase,QrYhVx4Ov4VvHkUysXoLNfCUHhx2,000f4c25-4e91-40d4-aaa1-b885692a0556,무료충전소,NaN,26.0,126.0,488.0,NaN
4415,0019f440-6bd4-4361-9f72-604d1a561933,2023-07-31 18:56:58,click_purchase,vOsUU67CntVzruO4TMvmk3oW5Qq1,0019f440-6bd4-4361-9f72-604d1a561933,1000 하트,NaN,95.0,424.0,242.0,NaN


In [27]:
event_df.loc[
    event_df["event_key"] == "click_purchase"
].shape[0]

13039

In [28]:
event_df.loc[
    event_df["event_key"] == "click_purchase",
    "item_name"
].isna().sum()

np.int64(0)

### item_name 결측 확인

- `item_name` 비결측 건수: 13,039건
- 비결측 데이터의 `event_key`: 모두 `click_purchase`
- `click_purchase` 전체 건수와 `item_name` 비결측 건수가 동일
- 따라서 `item_name`의 결측은 구매 클릭 이벤트 외에는 해당 값이 필요하지 않아 발생하는 구조적 결측으로 판단
- 별도의 결측치 처리 없이 유지

In [19]:
event_df.loc[
    event_df["event_key"] == "click_purchase",
    "item_name"
].value_counts(dropna=False)

item_name
무료충전소      7617
777 하트     3258
1000 하트    1112
200 하트     1001
4000 하트      51
Name: count, dtype: int64

---

In [24]:
event_df.loc[
    event_df["page_name"].notna(),
    "event_key"
].value_counts()

event_key
click_notice_detail     229358
click_question_start    220385
click_attendance        157737
click_question_ask      136766
view_signup              25630
click_profile_ask        14627
click_friend_invite       3221
click_invite_friend       1054
click_notice                 1
Name: count, dtype: int64

In [15]:
event_df[~event_df["page_name"].isna()].head()

,event_id,event_datetime,event_key,session_id,id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
24,000027a6-758c-41ec-b692-072ad06dd88e,2023-08-10 01:53:45,click_notice_detail,05iSJgKqQBgfPBveobPY1af1lRz1,000027a6-758c-41ec-b692-072ad06dd88e,NaN,notice,30.0,1612.0,7641.0,NaN
71,000065cc-d161-4fbe-8ccc-ce0142137f55,2023-08-09 19:15:22,click_notice_detail,f0FIPZL5lucXcGFR9zlZB5ArSrh1,000065cc-d161-4fbe-8ccc-ce0142137f55,NaN,notice,20.0,22.0,546.0,NaN
93,00008466-74af-4275-ac24-bb99aa01e79d,2023-07-21 18:53:17,click_attendance,7L8pmxIOo1SjlIxx59KXoyva2n63,00008466-74af-4275-ac24-bb99aa01e79d,NaN,home,86.0,448.0,4309.0,NaN
115,0000a3e9-3f69-4490-9cce-8ca8e5c1b306,2023-08-01 00:45:12,click_question_ask,01aed1ce-2515-4b1d-abe3-395b158cb8e8,0000a3e9-3f69-4490-9cce-8ca8e5c1b306,NaN,home,64.0,550.0,1394.0,NaN
140,0000c7b8-e7c2-4449-a358-093b082b789b,2023-08-05 15:11:43,click_attendance,il9yEXEYEffsi0bpkY1VrrFK8H82,0000c7b8-e7c2-4449-a358-093b082b789b,NaN,home,94.0,524.0,1717.0,NaN


In [29]:
page_events = event_df.loc[
    event_df["page_name"].notna(),
    "event_key"
].unique()

result = []

for key in page_events:
    total = (event_df["event_key"] == key).sum()
    non_null = event_df.loc[
        event_df["event_key"] == key,
        "page_name"
    ].notna().sum()

    result.append({
        "event_key": key,
        "total_count": total,
        "page_name_non_null": non_null,
        "missing_count": total - non_null
    })

pd.DataFrame(result).sort_values("event_key")

,event_key,total_count,page_name_non_null,missing_count
1,click_attendance,157737,157737,0
6,click_friend_invite,3221,3221,0
7,click_invite_friend,1054,1054,0
8,click_notice,1,1,0
0,click_notice_detail,229358,229358,0
4,click_profile_ask,14627,14627,0
2,click_question_ask,136766,136766,0
3,click_question_start,220385,220385,0
5,view_signup,25630,25630,0


### page_name 결측 확인

- `page_name`은 일부 이벤트에서만 사용되는 컬럼
- 값이 존재하는 이벤트:
  - `click_notice_detail` : 알림 센터 디테일에서 특정 알림을 클릭
  - `click_question_start` : 홈화면에서 질문 start 클릭
  - `click_attendance` : 출석체크 클릭
  - `click_question_ask` : 홈화면에서 ask 클릭
  - `view_signup` : 회원가입 페이지 진입
  - `click_profile_ask` : 친구 프로필에서 ‘친구에게 글을 남길까?’ 라는 버튼 클릭
  - `click_friend_invite` : 질문 대기상태에서 친구 초대하고 바로 받기 클릭
  - `click_invite_friend` : ask에서 친구 초대하기 클릭
  - `click_notice` : 알림 센터 클릭
- 위 이벤트들에서는 `page_name` 결측이 모두 0건
- 따라서 다른 이벤트의 `page_name` 결측은 데이터 누락이 아니라 구조적 결측으로 판단
- 별도 결측치 처리 없이 유지

In [ ]:
# page_name에 뭐 있는지 궁금해서 확인
page_name_counts = (
    event_df["page_name"]
    .dropna()
    .value_counts()
)

page_name_counts

page_name
home       518348
notice     229358
profile     14627
학년선택        10714
반선택          8592
학교선택         1904
번호인증         1246
아이디입력        1072
성별선택         1057
프사설정         1020
invite        816
이름입력           25
Name: count, dtype: int64

---

In [25]:
event_df.loc[
    event_df["question_id"].notna(),
    "event_key"
].value_counts()

event_key
skip_question    449484
Name: count, dtype: int64

In [31]:
skip_total = (
    event_df["event_key"] == "skip_question"
).sum()

skip_question_id_null = event_df.loc[
    event_df["event_key"] == "skip_question",
    "question_id"
].isna().sum()

print("skip_question 전체 건수:", skip_total)
print("skip_question 중 question_id 결측:", skip_question_id_null)

skip_question 전체 건수: 454981
skip_question 중 question_id 결측: 5497


- question_id는 skip_question 이벤트에서만 사용되는 컬럼이지만, skip_question 내에서도 일부 결측이 존재한다.
- 따라서 대부분은 구조적 결측이지만, 5,497건은 실제 누락 가능성이 있는 결측으로 별도 확인이 필요하다.

In [32]:
missing_qid = event_df[
    (event_df["event_key"] == "skip_question") &
    (event_df["question_id"].isna())
]

missing_qid.shape

(5497, 11)

In [38]:
# question_id가 있는 skip 이벤트만
skip_q = event_df.loc[
    event_df["question_id"].notna(),
    ["question_id"]
].drop_duplicates()

# polls_question 불러오기
question_df = pd.read_sql(
    "SELECT id, question_text FROM polls_question",
    engine
)

skip_q["question_id"] = skip_q["question_id"].astype("Int64")
question_df["id"] = question_df["id"].astype("Int64")

matched = skip_q.merge(
    question_df,
    left_on="question_id",
    right_on="id",
    how="left"
)

print("Hackle question_id 고유 개수:", len(skip_q))
print("polls_question 매칭 개수:", matched["id"].notna().sum())
print("미매칭 개수:", matched["id"].isna().sum())

Hackle question_id 고유 개수: 3897
polls_question 매칭 개수: 3897
미매칭 개수: 0


### question_id 결측 확인

- `question_id`는 `polls_question.id`를 참조하는 질문 식별자
- `hackle_events`에서는 `skip_question` 이벤트에서만 값이 기록됨
- 즉, 사용자가 어떤 질문을 스킵했는지 식별하기 위한 값으로 활용됨
- `skip_question` 전체 454,981건 중 5,497건에서 `question_id` 결측 확인
- `skip_question` 내 결측률은 약 1.21%
- `question_id`는 질문을 식별하는 ID이므로 0, 평균값, 최빈값 등으로 임의 대체하지 않음
- 전체 사용자 행동 로그 분석에서는 해당 행을 유지
- 질문별 스킵 횟수, 질문별 스킵률 등 `question_id`가 필요한 분석에서는 결측 행만 제외

---

In [44]:
friend_summary = (
    event_df
    .groupby("event_key")
    .agg(
        total_count=("event_key", "size"),
        friend_count_non_null=("friend_count", "count")
    )
)

friend_summary["missing_count"] = (
    friend_summary["total_count"]
    - friend_summary["friend_count_non_null"]
)

friend_summary["missing_ratio"] = (
    friend_summary["missing_count"]
    / friend_summary["total_count"]
    * 100
).round(2)

friend_summary.sort_values(
    "missing_count",
    ascending=False
)

,total_count,friend_count_non_null,missing_count,missing_ratio
event_key,,,,
$session_start,1036852,699143,337709,32.57
launch_app,986388,784486,201902,20.47
$session_end,649658,478267,171391,26.38
view_login,49275,30312,18963,38.48
view_signup,25630,8886,16744,65.33
view_home_tap,5392,17,5375,99.68
button,428,3,425,99.3
view_timeline_tap,1194508,1194493,15,0.0
view_questions_tap,353400,353386,14,0.0


### friend_count 결측 확인

- `friend_count`는 이벤트 발생 시점의 친구 수를 나타내는 컬럼
- 대부분의 주요 사용자 행동 이벤트에서는 결측 없이 기록됨
- 다만 `$session_start`, `launch_app`, `$session_end`, `view_login`, `view_signup` 등 일부 이벤트에서 높은 결측률이 확인됨
- 특히 세션 시작/종료, 로그인/회원가입 계열 이벤트에서는 `friend_count`가 선택적으로 기록되거나 수집되지 않은 것으로 판단
- `friend_count = 0`은 실제 친구 수가 0명이라는 의미이므로 결측값을 0으로 대체하지 않음
- 결측이 포함된 이벤트 행 자체는 사용자 행동 분석에 필요할 수 있으므로 삭제하지 않고 유지
- 필요 시 `friend_count`를 사용하는 분석에서만 비결측 행을 대상으로 활용

---

In [45]:
votes_summary = (
    event_df
    .groupby("event_key")
    .agg(
        total_count=("event_key", "size"),
        votes_count_non_null=("votes_count", "count")
    )
)

votes_summary["missing_count"] = (
    votes_summary["total_count"]
    - votes_summary["votes_count_non_null"]
)

votes_summary["missing_ratio"] = (
    votes_summary["missing_count"]
    / votes_summary["total_count"]
    * 100
).round(2)

votes_summary[
    votes_summary["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

,total_count,votes_count_non_null,missing_count,missing_ratio
event_key,,,,
$session_start,1036852,698239,338613,32.66
launch_app,986388,783942,202446,20.52
$session_end,649658,477800,171858,26.45
view_login,49275,30268,19007,38.57
view_signup,25630,8864,16766,65.42
view_home_tap,5392,0,5392,100.0
button,428,3,425,99.3
view_timeline_tap,1194508,1194493,15,0.0
view_questions_tap,353400,353386,14,0.0


### votes_count 결측 확인

- `votes_count`는 이벤트 발생 시점의 받은 투표 수를 나타내는 컬럼
- 대부분의 주요 사용자 행동 이벤트에서는 결측 없이 기록됨
- 다만 `$session_start`, `launch_app`, `$session_end`, `view_login`, `view_signup` 등 일부 이벤트에서 높은 결측률이 확인됨
- 특히 세션 시작/종료, 로그인/회원가입 계열 이벤트에서는 `votes_count`가 선택적으로 기록되거나 수집되지 않은 것으로 판단
- `votes_count = 0`은 실제 받은 투표 수가 0이라는 의미이므로 결측값을 0으로 대체하지 않음
- 결측이 포함된 이벤트 행은 삭제하지 않고 유지
- `votes_count`가 필요한 분석에서는 비결측 행을 기준으로 활용

---

In [46]:
heart_summary = (
    event_df
    .groupby("event_key")
    .agg(
        total_count=("event_key", "size"),
        heart_balance_non_null=("heart_balance", "count")
    )
)

heart_summary["missing_count"] = (
    heart_summary["total_count"]
    - heart_summary["heart_balance_non_null"]
)

heart_summary["missing_ratio"] = (
    heart_summary["missing_count"]
    / heart_summary["total_count"]
    * 100
).round(2)

heart_summary[
    heart_summary["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

,total_count,heart_balance_non_null,missing_count,missing_ratio
event_key,,,,
$session_start,1036852,709872,326980,31.54
launch_app,986388,791052,195336,19.8
$session_end,649658,483836,165822,25.52
view_login,49275,30923,18352,37.24
view_signup,25630,9159,16471,64.26
view_home_tap,5392,182,5210,96.62
button,428,3,425,99.3
view_timeline_tap,1194508,1194493,15,0.0
view_questions_tap,353400,353386,14,0.0


### heart_balance 결측 확인

- `heart_balance`는 이벤트 발생 시점의 잔여 하트 수를 나타내는 컬럼
- 대부분의 주요 사용자 행동 이벤트에서는 결측 없이 기록됨
- 다만 `$session_start`, `launch_app`, `$session_end`, `view_login`, `view_signup` 등 일부 이벤트에서 높은 결측률이 확인됨
- 세션 시작/종료, 로그인/회원가입 계열 이벤트에서는 `heart_balance`가 선택적으로 기록되거나 수집되지 않은 것으로 판단
- `heart_balance = 0`은 실제 잔여 하트가 0개라는 의미이므로 결측값을 0으로 대체하지 않음
- 결측이 포함된 이벤트 행은 삭제하지 않고 유지
- `heart_balance`가 필요한 분석에서는 비결측 행을 기준으로 활용

---

## 중복값 처리

In [47]:
event_df.duplicated().sum()

np.int64(0)

In [49]:
event_df["id"].duplicated().sum()

np.int64(0)

In [50]:
event_df["event_id"].duplicated().sum()

np.int64(0)

In [51]:
print(event_df["event_id"].nunique())
print(event_df["id"].nunique())
print(len(event_df))

11441319
11441319
11441319


In [52]:
event_df[["event_id", "id"]].head(20)

,event_id,id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,00000533-3f1c-4b3b-81f1-0c8f35754b4e
1,00000716-27e9-4e72-a602-d0ce61784b06,00000716-27e9-4e72-a602-d0ce61784b06
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc
3,00000981-5e2a-4111-993e-4f1891ad9a53,00000981-5e2a-4111-993e-4f1891ad9a53
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,00000a7a-ba72-4332-b4a9-7910670aaeb2
5,00000bfa-93ff-41d0-9b3a-c59ca76c0599,00000bfa-93ff-41d0-9b3a-c59ca76c0599
6,00000c49-cb9e-49d3-8d3d-cb2d1df8ed9c,00000c49-cb9e-49d3-8d3d-cb2d1df8ed9c
7,00000d23-8055-4534-893b-bf17425ab41e,00000d23-8055-4534-893b-bf17425ab41e
8,00001321-c6aa-4d28-ad3d-55aa4450aad7,00001321-c6aa-4d28-ad3d-55aa4450aad7
9,000015ae-60e0-495f-a356-9c1aaa18e841,000015ae-60e0-495f-a356-9c1aaa18e841


In [53]:
(event_df["event_id"] == event_df["id"]).all()

np.True_

In [54]:
# event_id와 id가 동일하므로 id 컬럼 제거
event_df = event_df.drop(columns=["id"])

### id 컬럼 확인

- `event_id`, `id` 모두 전체 11,441,319건에서 고유값을 가짐
- 두 컬럼의 값을 전체 행 기준으로 비교한 결과 완전히 동일함
- 동일한 이벤트 식별 정보가 중복 저장된 컬럼으로 판단
- `event_id`를 이벤트 식별자로 유지하고, 중복 컬럼인 `id`는 제거

---

In [56]:
event_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11441319 entries, 0 to 11441318
Data columns (total 10 columns):
 #   Column          Dtype         
---  ------          -----         
 0   event_id        str           
 1   event_datetime  datetime64[us]
 2   event_key       str           
 3   session_id      str           
 4   item_name       str           
 5   page_name       str           
 6   friend_count    Int64         
 7   votes_count     Int64         
 8   heart_balance   Int64         
 9   question_id     float64       
dtypes: Int64(3), datetime64[us](1), float64(1), str(5)
memory usage: 905.6 MB


In [57]:
event_df.head()

,event_id,event_datetime,event_key,session_id,item_name,page_name,friend_count,votes_count,heart_balance,question_id
0,00000533-3f1c-4b3b-81f1-0c8f35754b4e,2023-07-18 19:40:17,$session_start,4OzYh3seq3VKytpSn5pvQkZNQii1,NaN,NaN,<NA>,<NA>,<NA>,NaN
1,00000716-27e9-4e72-a602-d0ce61784b06,2023-07-18 21:07:24,click_question_open,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,NaN,NaN,64,436,4830,NaN
2,000007c8-68ce-40e6-9b1e-f0e34e8ff9cc,2023-08-06 20:18:03,click_bottom_navigation_profile,6bcea65d-9f40-46fc-888c-700fe707483f,NaN,NaN,26,174,4729,NaN
3,00000981-5e2a-4111-993e-4f1891ad9a53,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,NaN,NaN,61,44,142,NaN
4,00000a7a-ba72-4332-b4a9-7910670aaeb2,2023-07-24 15:03:37,click_bottom_navigation_lab,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,NaN,NaN,119,545,3287,NaN


In [59]:
print("최초 이벤트:", event_df["event_datetime"].min())
print("최종 이벤트:", event_df["event_datetime"].max())

최초 이벤트: 2023-07-18 00:00:00
최종 이벤트: 2023-08-10 23:59:59


### event_datetime 기간 확인

- `event_datetime`의 날짜 범위를 확인한 결과, Hackle 이벤트 데이터는 **2023-07-18 ~ 2023-08-10** 기간으로 구성됨
- 총 24일의 이벤트 로그이며, 분석 시 해당 기간을 기준으로 활용
- `event_datetime`은 이미 `datetime64[us]` 형식이므로 별도 타입 변환은 필요하지 않음

---

## 이상치 처리

In [61]:
event_df[
    ["friend_count", "votes_count", "heart_balance"]
].describe()

,friend_count,votes_count,heart_balance
count,10688763.0,10686765.0,10712676.0
mean,54.343567,257.274234,16269.292655
std,33.507977,218.068191,3317340.091112
min,0.0,0.0,0.0
25%,32.0,97.0,434.0
50%,49.0,210.0,1249.0
75%,71.0,362.0,3188.0
max,1365.0,3017.0,884999804.0


In [64]:
event_df["heart_balance"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
    0.9999
])

0.9000      6518
0.9500     10120
0.9900     20073
0.9950     25200
0.9990     38037
0.9999    263783
Name: heart_balance, dtype: Int64

In [65]:
for threshold in [
    10_000,
    50_000,
    100_000,
    1_000_000,
    10_000_000
]:
    count = (event_df["heart_balance"] > threshold).sum()
    print(f"{threshold:,} 초과: {count:,}건")

10,000 초과: 546,826건
50,000 초과: 6,097건
100,000 초과: 1,605건
1,000,000 초과: 1,028건
10,000,000 초과: 160건


### heart_balance 이상치 확인

- `heart_balance`의 중앙값은 1,249, 99.9% 분위수는 38,037
- 최대값은 884,999,804로 일반적인 분포와 매우 큰 차이를 보임
- 1,000,000 초과 값은 1,028건으로 확인됨
- 따라서 1,000,000 초과 값을 극단 이상치 후보로 표시
- 원본 데이터는 유지하고 `heart_balance_outlier` 플래그를 생성
- 이후 사용자/세션 단위 확인 후 실제 오류값으로 판단되면 분석에서 제외

In [66]:
event_df["friend_count"].quantile([
    0.90, 0.95, 0.99, 0.999
])

0.900     95
0.950    112
0.990    155
0.999    249
Name: friend_count, dtype: Int64

In [67]:
event_df["votes_count"].quantile([
    0.90, 0.95, 0.99, 0.999
])

0.900     536
0.950     663
0.990     982
0.999    1416
Name: votes_count, dtype: Int64

### friend_count / votes_count 이상치 확인

- `friend_count`
  - 99% 분위수: 155
  - 99.9% 분위수: 249
  - 최대값: 1,365
- `votes_count`
  - 99% 분위수: 982
  - 99.9% 분위수: 1,416
  - 최대값: 3,017
- 상위 값이 존재하지만 `heart_balance`처럼 비정상적으로 극단적인 분포는 아님
- 현재 단계에서는 실제 사용자 활동 범위 내 값일 가능성이 있다고 판단하여 별도 제거하지 않고 유지

---
---
---

# hackle_properties 테이블 전처리

In [68]:
property_df = pd.read_sql(
    "SELECT * FROM hackle_properties",
    engine
)